In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.append(PROJECT_ROOT)

import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
from data import MessageDataset, setup_tokenizer, get_json_files, collate_fn, create_input_target_pairs

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
setup_tokenizer(tokenizer)

json_files = get_json_files(os.path.join(PROJECT_ROOT, "data"))
dataset = MessageDataset(json_files, tokenizer=tokenizer, sequence_length=256)
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(batch, tokenizer.pad_token_id),
)

VOCAB_SIZE = len(tokenizer)
SEQUENCE_LENGTH = 256
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Vocab size: {VOCAB_SIZE}")
print(f"Dataset: {len(dataset)} sequences | {len(dataloader)} batches")

Vocab size: 50266
Dataset: 16838 sequences | 2105 batches


# Model

In [3]:
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import MultiheadAttention

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self, d_model, max_sequence_length):
        super().__init__()
        self.Q = nn.Linear(d_model, d_model)
        self.K = nn.Linear(d_model, d_model)
        self.V = nn.Linear(d_model, d_model)

        # Causal Mask -> can't attend to future tokens
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(max_sequence_length, max_sequence_length))
        )

    def forward(self, x):
        q = self.Q(x)
        k = self.K(x)
        v = self.V(x)

        # Scaled dot-product attention
        d_k = q.size(-1)
        scores = (q @ k.transpose(-2, -1)) / d_k ** 0.5

        # Apply causal mask
        seq_len = x.size(-2)
        scores = scores.masked_fill(self.mask[:seq_len, :seq_len] == 0, float('-inf'))

        weights = F.softmax(scores, dim=-1)
        return weights @ v

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, 
        d_model, 
        num_attention_heads,
        max_sequence_length
    ):
        super().__init__()
        # Layer Norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Multi-head Attention
        self.head_dim = d_model // num_attention_heads
        self.attention_heads = nn.ModuleList(
            AttentionHead(
                self.head_dim,
                max_sequence_length
            ) for _ in range(num_attention_heads)
        ) 

        # Projection Layer - after concatting attention
        self.projection = nn.Linear(d_model, d_model)

        # Feed Forward
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
        
    def forward(self, x):
        
        out = self.norm1(x)
        
        head_output = []
        for i, head in enumerate(self.attention_heads):
            out_slice = out[:, :, i * self.head_dim: (i+1) * self.head_dim]
            head_output.append(head(out_slice))

        out = torch.cat(head_output, dim=-1)
        out = self.projection(out)
        
        x = x + out

        x = x + self.ff(self.norm2(x))
        
        return x

In [ ]:
class GPT2Small(nn.Module):
    def __init__(self, 
        vocab_size=VOCAB_SIZE, 
        d_model=768,
        max_sequence_length=256, 
        num_transformer_blocks=12, 
        num_attention_heads=12
    ):
        super().__init__()
        self.max_sequence_length = max_sequence_length

        self.token_emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
        self.pos_emb = nn.Embedding(num_embeddings=max_sequence_length, embedding_dim=d_model)
        
        self.transformer_blocks = nn.ModuleList(
            TransformerBlock(
                d_model,
                num_attention_heads,
                max_sequence_length
            ) 
            for _ in range(num_transformer_blocks)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        
        

    def forward(self, x):
        # token embedding
        x = self.token_emb(x) + self.pos_emb(torch.arange(x.size(1), device=x.device))

        for i, transformer in enumerate(self.transformer_blocks):
            x = transformer(x)
        
        x = self.norm1(x)
        x = self.lm_head(x)

        return x

In [ ]:
# Training Loop